# 0. Imports & Configs

In [ ]:
# All imports
import torch
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import os
import timm
import time
import json

import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from ghost_module import GhostModule
from cbam_module import CBAM

from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
import seaborn as sns
import numpy as np

print(f"CUDA is available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"Number of CUDA devices: {torch.cuda.device_count()}")
print(f"Current device: {torch.cuda.current_device()}, Device name: {torch.cuda.get_device_name(0)}")
print(f"Device capability: {torch.cuda.get_device_capability(0)}")
print(f"Device properties: {torch.cuda.get_device_properties(0)}")

In [ ]:
# Configuration
BASE_FOLDER = "/home/J/research/asd/train"
DATASET_PATH = os.path.join(BASE_FOLDER, "mahmoud-dataset/Images")
MODEL_SAVE_PATH = os.path.join(BASE_FOLDER, "ghost+cbam/convnexttiny_v2")
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_PATH, "best.pth")

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"Base folder: {BASE_FOLDER}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Model save path: {MODEL_SAVE_PATH}")

In [ ]:
num_epochs = 30
criterion = torch.nn.BCEWithLogitsLoss()
patience = 25

# 1. Load Data

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
targets = [label for _, label in dataset]

train_idx, temp_idx = train_test_split(
    range(len(dataset)),
    test_size=0.3,
    stratify=targets,
    random_state=42,
)

temp_targets = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.385,
    stratify=temp_targets,
    random_state=42,
)

train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
val_sampler   = torch.utils.data.SubsetRandomSampler(val_idx)
test_sampler  = torch.utils.data.SubsetRandomSampler(test_idx)

batch_size = 32

train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
val_loader   = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=val_sampler)
test_loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=test_sampler)

print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
print(f"Train batches: {len(train_loader)},  Val batches: {len(val_loader)},   Test batches: {len(test_loader)}")

In [ ]:
def count_classes(indices, dataset):
    labels = [dataset.samples[i][1] for i in indices]
    return Counter(labels)

print("Class mapping:", dataset.class_to_idx)
print("Train class count:", count_classes(train_idx, dataset))
print("Val class count:  ", count_classes(val_idx, dataset))
print("Test class count: ", count_classes(test_idx, dataset))

# 2. Augment Data

In [ ]:
def augment_pre_fmmix1(images, labels):
    B = images.size(0)
    flip_mask = torch.rand(B, device=images.device) < 0.5
    for i in range(B):
        if flip_mask[i]:
            images[i] = torch.flip(images[i], dims=[2])
    brightness_delta = torch.empty(B, 1, 1, 1, device=images.device).uniform_(-0.1, 0.1)
    images = images + brightness_delta
    mean = images.mean(dim=(2, 3), keepdim=True)
    contrast_factor = torch.empty(B, 1, 1, 1, device=images.device).uniform_(0.9, 1.1)
    images = (images - mean) * contrast_factor + mean
    images = torch.clamp(images, 0.0, 1.0)
    return images, labels

class Args:
    def __init__(self, alpha=0.05, mask_area_mode=1):
        self.alpha = alpha
        self.mask_area_mode = mask_area_mode

def fmmix1(args, x, target):
    B, C, H, W = x.shape
    shuffle_idx = torch.randperm(B)
    x_shuffle = x[shuffle_idx]
    max_val_indices = torch.argmax(x_shuffle.view(B, C, -1), dim=2)
    max_indices_unraveled = torch.stack((max_val_indices // W, max_val_indices % W), dim=-1)
    
    lambdas = torch.sqrt(args.alpha * torch.rand(B, 1, 1, 1, device=x.device))
    Wb = (lambdas * W).squeeze(-1)
    Hb = (lambdas * H).squeeze(-1)
    
    x_range = torch.arange(W, device=x.device)[None, :].expand(B, C, H, W)
    y_range = torch.arange(H, device=x.device)[:, None].expand(B, C, H, W)
    x_min = torch.clamp(max_indices_unraveled[..., 1][..., None, None] - Wb[..., None] // 2, 0, W)
    y_min = torch.clamp(max_indices_unraveled[..., 0][..., None, None] - Hb[..., None] // 2, 0, H)
    x_max = torch.clamp(x_min + Wb[..., None], 0, W)
    y_max = torch.clamp(y_min + Hb[..., None], 0, H)
    masks = (x_range >= x_min) & (x_range < x_max) & (y_range >= y_min) & (y_range < y_max)
    x = x*~masks + x_shuffle*masks
    
    target_shuffled = target[shuffle_idx]
    p_src = torch.sum(masks, dim=[1,2,3])/(C*W*H)
    p_tar = 1 - p_src
    return x, [2, [p_tar, p_src], [target, target_shuffled]]

# 3. Build Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("convnextv2_tiny.fcmae", pretrained=True, num_classes=0, global_pool="")
        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        self.head = torch.nn.Linear(self.backbone.num_features, 1)
    
    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.gap(x).flatten(1)
        return self.head(x)

model = Model().to(device)
print(f"Model created. Backbone features: {model.backbone.num_features}")

# Phương án 1: GhostModule Head + CBAM

CBAM được đặt sau backbone để refine features trước khi vào GhostModule head.

## Build Model - PA1 + CBAM

In [ ]:
class ModelPA1_CBAM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("convnextv2_tiny.fcmae", pretrained=True, num_classes=0, global_pool="")
        
        # CBAM sau backbone
        in_features = self.backbone.num_features
        self.cbam = CBAM(in_features, reduction=16)
        
        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        
        # GhostModule head
        mid_features = in_features // 2
        self.ghost_head = torch.nn.Sequential(
            GhostModule(in_features, mid_features, kernel_size=1, ratio=2),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(mid_features, 1)
        )
    
    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.cbam(x)  # CBAM refine features
        x = self.gap(x).flatten(1)
        x = x.unsqueeze(-1).unsqueeze(-1)
        x = self.ghost_head[0](x)
        x = x.flatten(1)
        x = self.ghost_head[1](x)
        x = self.ghost_head[2](x)
        return x

model_pa1_cbam = ModelPA1_CBAM().to(device)
print(f"PA1+CBAM Model. Features: {model_pa1_cbam.backbone.num_features}")

## Train Model - PA1 + CBAM

In [ ]:
optimizer_pa1 = torch.optim.Adam(model_pa1_cbam.parameters(), lr=1e-4)
scheduler_pa1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pa1, T_max=100, eta_min=1e-6)

best_val_loss_pa1 = float('inf')
patience_counter_pa1 = 0
train_losses_pa1, val_losses_pa1 = [], []
train_accs_pa1, val_accs_pa1 = [], []

for epoch in range(num_epochs):
    start_time = time.time()
    model_pa1_cbam.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        images_aug, labels_aug = augment_pre_fmmix1(images.clone(), labels.clone())
        
        optimizer_pa1.zero_grad()
        outputs = model_pa1_cbam(images_aug).squeeze(1)
        loss = criterion(outputs, labels_aug)
        loss.backward()
        optimizer_pa1.step()
        
        with torch.no_grad():
            train_correct += ((outputs > 0).int() == labels_aug.int()).sum().item()
        train_loss += loss.item()
        train_total += labels.size(0)
    
    train_acc = train_correct / train_total
    train_loss /= len(train_loader)
    train_accs_pa1.append(train_acc)
    train_losses_pa1.append(train_loss)
    
    model_pa1_cbam.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model_pa1_cbam(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()
            val_correct += ((torch.sigmoid(outputs) > 0.5).int() == labels.int()).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    val_loss /= len(val_loader)
    val_accs_pa1.append(val_acc)
    val_losses_pa1.append(val_loss)
    
    print(f"[PA1+CBAM] Epoch {epoch+1}/{num_epochs} | Train: Loss={train_loss:.4f} Acc={train_acc:.4f} | Val: Loss={val_loss:.4f} Acc={val_acc:.4f} | Time={time.time()-start_time:.1f}s")
    
    scheduler_pa1.step()
    
    if val_loss < best_val_loss_pa1:
        best_val_loss_pa1 = val_loss
        torch.save(model_pa1_cbam.state_dict(), os.path.join(MODEL_SAVE_PATH, 'best_pa1_cbam.pth'))
        patience_counter_pa1 = 0
        print(f"  ✓ PA1+CBAM best model saved")
    else:
        patience_counter_pa1 += 1
        if patience_counter_pa1 >= patience:
            print(f"PA1+CBAM early stopping at epoch {epoch+1}")
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses_pa1, label='Train'); ax1.plot(val_losses_pa1, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('PA1+CBAM Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(train_accs_pa1, label='Train'); ax2.plot(val_accs_pa1, label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('PA1+CBAM Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## Evaluate Model - PA1 + CBAM

In [ ]:
model_pa1_cbam.load_state_dict(torch.load(os.path.join(MODEL_SAVE_PATH, 'best_pa1_cbam.pth')))
model_pa1_cbam.eval()

y_all_pa1, yhat_all_pa1 = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = torch.sigmoid(model_pa1_cbam(images.to(device)))
        yhat_all_pa1.extend((outputs > 0.5).int().squeeze(1).cpu().numpy())
        y_all_pa1.extend(labels.numpy())

y_all_pa1, yhat_all_pa1 = np.array(y_all_pa1), np.array(yhat_all_pa1)
accuracy_pa1 = (y_all_pa1 == yhat_all_pa1).mean()
precision_pa1 = precision_score(y_all_pa1, yhat_all_pa1, average='weighted')
recall_pa1 = recall_score(y_all_pa1, yhat_all_pa1, average='weighted')

print(f"[PA1+CBAM] Test Accuracy:  {accuracy_pa1:.4f}")
print(f"[PA1+CBAM] Test Precision: {precision_pa1:.4f}")
print(f"[PA1+CBAM] Test Recall:    {recall_pa1:.4f}")
print("\n" + classification_report(y_all_pa1, yhat_all_pa1, target_names=dataset.classes))

conf_mat_pa1 = confusion_matrix(1 - y_all_pa1, 1 - yhat_all_pa1)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_mat_pa1, annot=True, fmt='g', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('PA1+CBAM Confusion Matrix')
ax.xaxis.set_ticklabels(['non-ASD', 'ASD']); ax.yaxis.set_ticklabels(['non-ASD', 'ASD'])
plt.show()

# Phương án 2: GhostBottleneck + CBAM

CBAM được đặt giữa ghost1 và ghost2 trong GhostBottleneck để refine intermediate features.

## Build Model - PA2 + CBAM

In [ ]:
class GhostBottleneck_CBAM(torch.nn.Module):
    def __init__(self, in_ch, out_ch, expansion=2):
        super().__init__()
        mid_ch = in_ch * expansion
        self.ghost1 = GhostModule(in_ch, mid_ch, kernel_size=1, ratio=2)
        self.cbam = CBAM(mid_ch, reduction=16)  # CBAM giữa 2 ghost
        self.ghost2 = GhostModule(mid_ch, out_ch, kernel_size=1, ratio=2, relu=False)
    
    def forward(self, x):
        x = self.ghost1(x)
        x = self.cbam(x)  # CBAM attention
        return self.ghost2(x)

class ModelPA2_CBAM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("convnextv2_tiny.fcmae", pretrained=True, num_classes=0, global_pool="")
        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        
        in_features = self.backbone.num_features
        self.ghost_bottleneck = torch.nn.Sequential(
            GhostBottleneck_CBAM(in_features, in_features // 2, expansion=2),
            torch.nn.ReLU(inplace=True),
            GhostBottleneck_CBAM(in_features // 2, in_features // 4, expansion=2),
            torch.nn.ReLU(inplace=True)
        )
        self.head = torch.nn.Linear(in_features // 4, 1)
    
    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.gap(x).flatten(1)
        x = x.unsqueeze(-1).unsqueeze(-1)
        x = self.ghost_bottleneck(x).flatten(1)
        return self.head(x)

model_pa2_cbam = ModelPA2_CBAM().to(device)
print(f"PA2+CBAM Model. Features: {model_pa2_cbam.backbone.num_features}")

## Train Model - PA2 + CBAM

In [ ]:
optimizer_pa2 = torch.optim.Adam(model_pa2_cbam.parameters(), lr=1e-4)
scheduler_pa2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pa2, T_max=100, eta_min=1e-6)

best_val_loss_pa2 = float('inf')
patience_counter_pa2 = 0
train_losses_pa2, val_losses_pa2, train_accs_pa2, val_accs_pa2 = [], [], [], []

for epoch in range(num_epochs):
    start_time = time.time()
    model_pa2_cbam.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        images_aug, labels_aug = augment_pre_fmmix1(images.clone(), labels.clone())
        
        optimizer_pa2.zero_grad()
        outputs = model_pa2_cbam(images_aug).squeeze(1)
        loss = criterion(outputs, labels_aug)
        loss.backward()
        optimizer_pa2.step()
        
        with torch.no_grad():
            train_correct += ((outputs > 0).int() == labels_aug.int()).sum().item()
        train_loss += loss.item()
        train_total += labels.size(0)
    
    train_acc = train_correct / train_total
    train_loss /= len(train_loader)
    train_accs_pa2.append(train_acc)
    train_losses_pa2.append(train_loss)
    
    model_pa2_cbam.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model_pa2_cbam(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()
            val_correct += ((torch.sigmoid(outputs) > 0.5).int() == labels.int()).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    val_loss /= len(val_loader)
    val_accs_pa2.append(val_acc)
    val_losses_pa2.append(val_loss)
    
    print(f"[PA2+CBAM] Epoch {epoch+1}/{num_epochs} | Train: Loss={train_loss:.4f} Acc={train_acc:.4f} | Val: Loss={val_loss:.4f} Acc={val_acc:.4f} | Time={time.time()-start_time:.1f}s")
    
    scheduler_pa2.step()
    
    if val_loss < best_val_loss_pa2:
        best_val_loss_pa2 = val_loss
        torch.save(model_pa2_cbam.state_dict(), os.path.join(MODEL_SAVE_PATH, 'best_pa2_cbam.pth'))
        patience_counter_pa2 = 0
        print(f"  ✓ PA2+CBAM best model saved")
    else:
        patience_counter_pa2 += 1
        if patience_counter_pa2 >= patience:
            print(f"PA2+CBAM early stopping at epoch {epoch+1}")
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses_pa2, label='Train'); ax1.plot(val_losses_pa2, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('PA2+CBAM Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(train_accs_pa2, label='Train'); ax2.plot(val_accs_pa2, label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('PA2+CBAM Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## Evaluate Model - PA2 + CBAM

In [ ]:
model_pa2_cbam.load_state_dict(torch.load(os.path.join(MODEL_SAVE_PATH, 'best_pa2_cbam.pth')))
model_pa2_cbam.eval()

y_all_pa2, yhat_all_pa2 = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = torch.sigmoid(model_pa2_cbam(images.to(device)))
        yhat_all_pa2.extend((outputs > 0.5).int().squeeze(1).cpu().numpy())
        y_all_pa2.extend(labels.numpy())

y_all_pa2, yhat_all_pa2 = np.array(y_all_pa2), np.array(yhat_all_pa2)
accuracy_pa2 = (y_all_pa2 == yhat_all_pa2).mean()
precision_pa2 = precision_score(y_all_pa2, yhat_all_pa2, average='weighted')
recall_pa2 = recall_score(y_all_pa2, yhat_all_pa2, average='weighted')

print(f"[PA2+CBAM] Test Accuracy:  {accuracy_pa2:.4f}")
print(f"[PA2+CBAM] Test Precision: {precision_pa2:.4f}")
print(f"[PA2+CBAM] Test Recall:    {recall_pa2:.4f}")
print("\n" + classification_report(y_all_pa2, yhat_all_pa2, target_names=dataset.classes))

conf_mat_pa2 = confusion_matrix(1 - y_all_pa2, 1 - yhat_all_pa2)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_mat_pa2, annot=True, fmt='g', cmap='Greens', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('PA2+CBAM Confusion Matrix')
ax.xaxis.set_ticklabels(['non-ASD', 'ASD']); ax.yaxis.set_ticklabels(['non-ASD', 'ASD'])
plt.show()

# Phương án 3: GhostAdapter + CBAM

CBAM được đặt sau GhostAdapter để refine output features của adapter.

## Build Model - PA3 + CBAM

In [ ]:
class GhostAdapter(torch.nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        mid_channels = in_channels // reduction
        self.adapter = torch.nn.Sequential(
            GhostModule(in_channels, mid_channels, kernel_size=1, ratio=2),
            torch.nn.ReLU(inplace=True),
            GhostModule(mid_channels, in_channels, kernel_size=1, ratio=2, relu=False)
        )
    
    def forward(self, x):
        return x + self.adapter(x)

class ModelPA3_CBAM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("convnextv2_tiny.fcmae", pretrained=True, num_classes=0, global_pool="")
        self.gap = torch.nn.AdaptiveAvgPool2d(1)
        
        in_features = self.backbone.num_features
        self.ghost_adapter = GhostAdapter(in_features, reduction=4)
        self.cbam = CBAM(in_features, reduction=16)  # CBAM sau adapter
        self.head = torch.nn.Linear(in_features, 1)
    
    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.ghost_adapter(x)
        x = self.cbam(x)  # CBAM refine adapter output
        x = self.gap(x).flatten(1)
        return self.head(x)

model_pa3_cbam = ModelPA3_CBAM().to(device)
print(f"PA3+CBAM Model. Features: {model_pa3_cbam.backbone.num_features}")

## Train Model - PA3 + CBAM

In [ ]:
optimizer_pa3 = torch.optim.Adam(model_pa3_cbam.parameters(), lr=1e-4)
scheduler_pa3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pa3, T_max=100, eta_min=1e-6)

best_val_loss_pa3 = float('inf')
patience_counter_pa3 = 0
train_losses_pa3, val_losses_pa3, train_accs_pa3, val_accs_pa3 = [], [], [], []

for epoch in range(num_epochs):
    start_time = time.time()
    model_pa3_cbam.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        images_aug, labels_aug = augment_pre_fmmix1(images.clone(), labels.clone())
        
        optimizer_pa3.zero_grad()
        outputs = model_pa3_cbam(images_aug).squeeze(1)
        loss = criterion(outputs, labels_aug)
        loss.backward()
        optimizer_pa3.step()
        
        with torch.no_grad():
            train_correct += ((outputs > 0).int() == labels_aug.int()).sum().item()
        train_loss += loss.item()
        train_total += labels.size(0)
    
    train_acc = train_correct / train_total
    train_loss /= len(train_loader)
    train_accs_pa3.append(train_acc)
    train_losses_pa3.append(train_loss)
    
    model_pa3_cbam.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model_pa3_cbam(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()
            val_correct += ((torch.sigmoid(outputs) > 0.5).int() == labels.int()).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    val_loss /= len(val_loader)
    val_accs_pa3.append(val_acc)
    val_losses_pa3.append(val_loss)
    
    print(f"[PA3+CBAM] Epoch {epoch+1}/{num_epochs} | Train: Loss={train_loss:.4f} Acc={train_acc:.4f} | Val: Loss={val_loss:.4f} Acc={val_acc:.4f} | Time={time.time()-start_time:.1f}s")
    
    scheduler_pa3.step()
    
    if val_loss < best_val_loss_pa3:
        best_val_loss_pa3 = val_loss
        torch.save(model_pa3_cbam.state_dict(), os.path.join(MODEL_SAVE_PATH, 'best_pa3_cbam.pth'))
        patience_counter_pa3 = 0
        print(f"  ✓ PA3+CBAM best model saved")
    else:
        patience_counter_pa3 += 1
        if patience_counter_pa3 >= patience:
            print(f"PA3+CBAM early stopping at epoch {epoch+1}")
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses_pa3, label='Train'); ax1.plot(val_losses_pa3, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('PA3+CBAM Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(train_accs_pa3, label='Train'); ax2.plot(val_accs_pa3, label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('PA3+CBAM Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## Evaluate Model - PA3 + CBAM

In [ ]:
model_pa3_cbam.load_state_dict(torch.load(os.path.join(MODEL_SAVE_PATH, 'best_pa3_cbam.pth')))
model_pa3_cbam.eval()

y_all_pa3, yhat_all_pa3 = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = torch.sigmoid(model_pa3_cbam(images.to(device)))
        yhat_all_pa3.extend((outputs > 0.5).int().squeeze(1).cpu().numpy())
        y_all_pa3.extend(labels.numpy())

y_all_pa3, yhat_all_pa3 = np.array(y_all_pa3), np.array(yhat_all_pa3)
accuracy_pa3 = (y_all_pa3 == yhat_all_pa3).mean()
precision_pa3 = precision_score(y_all_pa3, yhat_all_pa3, average='weighted')
recall_pa3 = recall_score(y_all_pa3, yhat_all_pa3, average='weighted')

print(f"[PA3+CBAM] Test Accuracy:  {accuracy_pa3:.4f}")
print(f"[PA3+CBAM] Test Precision: {precision_pa3:.4f}")
print(f"[PA3+CBAM] Test Recall:    {recall_pa3:.4f}")
print("\n" + classification_report(y_all_pa3, yhat_all_pa3, target_names=dataset.classes))

conf_mat_pa3 = confusion_matrix(1 - y_all_pa3, 1 - yhat_all_pa3)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(conf_mat_pa3, annot=True, fmt='g', cmap='Oranges', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('PA3+CBAM Confusion Matrix')
ax.xaxis.set_ticklabels(['non-ASD', 'ASD']); ax.yaxis.set_ticklabels(['non-ASD', 'ASD'])
plt.show()